In [5]:
!pip install fabric -q

You should consider upgrading via the '/Users/anokhimehta/opt/anaconda3/bin/python -m pip install --upgrade pip' command.


In [6]:
from fabric import Connection
import os

FLOATING_IP = "129.114.108.98"  # update if IP changes

conn = Connection(
    host=FLOATING_IP,
    user="cc",
    connect_kwargs={"key_filename": os.path.expanduser("~/.ssh/id_rsa_chameleon")}
)

# Test connection
conn.run("whoami && hostname")

ModuleNotFoundError: No module named 'fabric'

In [ ]:
conn.run("cd ~/bestshot && git pull")

In [ ]:
conn.run("""
cd ~/bestshot && docker run --rm \
    -v ~/bestshot:/workspace \
    -w /workspace/training \
    --env-file /home/cc/bestshot/serving/.env \
    ghcr.io/anokhimehta/bestshot-training:latest \
    python retrain.py
""")

In [ ]:
conn.run("""
cd ~/bestshot && docker run --rm \
    -v ~/bestshot:/workspace \
    -w /workspace/training \
    --env-file /home/cc/bestshot/serving/.env \
    ghcr.io/anokhimehta/bestshot-training:latest \
    python retrain.py --force
""")

In [ ]:
conn.run("""
python3 -c "
import swiftclient, os, json
from dotenv import load_dotenv
load_dotenv('/home/cc/bestshot/serving/.env')
conn = swiftclient.Connection(
    auth_version='3',
    authurl=os.environ['OS_AUTH_URL'],
    os_options={
        'application_credential_id': os.environ['OS_APPLICATION_CREDENTIAL_ID'],
        'application_credential_secret': os.environ['OS_APPLICATION_CREDENTIAL_SECRET'],
        'region_name': os.environ['OS_REGION_NAME'],
        'auth_type': 'v3applicationcredential'
    }
)
_, content = conn.get_object(os.environ['BUCKET_NAME'], 'training/last_retrain.json')
print(json.loads(content))
"
""")

In [ ]:
conn.run("kubectl get jobs -n bestshot-platform")